# Creating Embeddings for the Maternal Health Knowledge Base

This notebook creates vector embeddings for the processed maternal health
knowledge base used in the Pregnancy Journey Partner chatbot.

## Purpose

The goal is to convert each text chunk into a numerical vector representation
that can later be used for semantic search and retrieval.

## Knowledge Base

- Organizations: WHO and UNICEF
- Documents: 14
- Pages extracted: 891
- Text chunks: 3,596
- Chunk size: 1,000 characters
- Chunk overlap: 200 characters

## Pipeline

Text chunks → Embedding Model → Embeddings → Vector Database → RAG Chatbot

In [1]:
import polars as pl
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Define the path to the processed knowledge base

chunks_path = Path("../knowledge_base/processed/chunks.parquet")

print("Knowledge base path:")
print(chunks_path)
print("File exists:", chunks_path.exists())

Knowledge base path:
..\knowledge_base\processed\chunks.parquet
File exists: True


In [3]:
# Load the processed knowledge base

chunks_df = pl.read_parquet(chunks_path)

print(f"Total chunks: {chunks_df.height}")
print(f"Total columns: {len(chunks_df.columns)}")
print("Columns:")
print(chunks_df.columns)

Total chunks: 3596
Total columns: 9
Columns:
['chunk_id', 'organization', 'document', 'page_number', 'chunk_number', 'text', 'source_id', 'title', 'url']


In [4]:
# Inspect the first few knowledge-base chunks

chunks_df.head(5)

chunk_id,organization,document,page_number,chunk_number,text,source_id,title,url
u32,str,str,i64,i64,str,i64,str,str
1,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…",10,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…"
2,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…",10,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…"
3,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…",10,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…"
4,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…",10,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…"
5,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,5,"""initial capital letters. All r…",10,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…"


In [5]:
from sentence_transformers import SentenceTransformer

print("Sentence Transformers imported successfully.")

Sentence Transformers imported successfully.


In [6]:
# Load the embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Ark of Designs\Desktop\Pregnancy-Journey-Partner\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ark of Designs\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Embedding dimension: 384


C:\Users\Ark of Designs\AppData\Local\Temp\ipykernel_15304\3998517131.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [7]:
# Test the embedding model on one knowledge-base chunk

sample_text = chunks_df.select("text").item(0, 0)

sample_embedding = embedding_model.encode(
    sample_text,
    convert_to_numpy=True
)

print("Embedding created successfully.")
print("Embedding shape:", sample_embedding.shape)
print("Embedding dimension:", len(sample_embedding))

Embedding created successfully.
Embedding shape: (384,)
Embedding dimension: 384


In [8]:
# Inspect the first 10 values of the embedding

print(sample_embedding[:10])

[-0.05874221 -0.00226979 -0.08372576  0.0099081   0.06697527  0.07904558
  0.00047689  0.0178018   0.02673725  0.02579554]


In [9]:
# Generate embeddings for all knowledge-base chunks

texts = chunks_df["text"].to_list()

print(f"Total texts to embed: {len(texts)}")

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbeddings created successfully.")
print("Number of embeddings:", embeddings.shape[0])
print("Embedding dimension:", embeddings.shape[1])

Total texts to embed: 3596


Batches:   0%|          | 0/113 [00:00<?, ?it/s]


Embeddings created successfully.
Number of embeddings: 3596
Embedding dimension: 384


In [10]:
# Verify the embedding matrix

print("Expected chunks:", chunks_df.height)
print("Actual embeddings:", embeddings.shape[0])
print("Embedding dimensions:", embeddings.shape[1])

assert embeddings.shape[0] == chunks_df.height
assert embeddings.shape[1] == 384

print("\n✓ Every chunk has a 384-dimensional embedding.")

Expected chunks: 3596
Actual embeddings: 3596
Embedding dimensions: 384

✓ Every chunk has a 384-dimensional embedding.


In [11]:
from pathlib import Path

# Define the output path
embeddings_path = Path("../knowledge_base/processed/embeddings.npy")

# Save the embeddings
import numpy as np

np.save(embeddings_path, embeddings)

print("Embeddings saved successfully.")
print(f"File: {embeddings_path}")
print(f"Shape: {embeddings.shape}")

Embeddings saved successfully.
File: ..\knowledge_base\processed\embeddings.npy
Shape: (3596, 384)


In [12]:
# Load the saved embeddings to verify the file

loaded_embeddings = np.load(embeddings_path)

print("Embeddings loaded successfully.")
print("Shape:", loaded_embeddings.shape)
print("Data type:", loaded_embeddings.dtype)

assert loaded_embeddings.shape == (3596, 384)

print("✓ Embeddings verified successfully.")

Embeddings loaded successfully.
Shape: (3596, 384)
Data type: float32
✓ Embeddings verified successfully.


In [13]:
# Verify that the number of embeddings matches the number of chunks

print("Chunks:", chunks_df.height)
print("Embeddings:", loaded_embeddings.shape[0])

assert chunks_df.height == loaded_embeddings.shape[0]

print("✓ Chunk and embedding counts match perfectly.")

Chunks: 3596
Embeddings: 3596
✓ Chunk and embedding counts match perfectly.


In [14]:
import chromadb

print("ChromaDB imported successfully.")

ChromaDB imported successfully.


In [15]:
chroma_client = chromadb.PersistentClient(
    path="../knowledge_base/vector_db"
)

print("ChromaDB client created successfully.")

ChromaDB client created successfully.


In [16]:
# Create a test collection

test_collection = chroma_client.get_or_create_collection(
    name="pregnancy_knowledge_test",
    metadata={"description": "Test collection for pregnancy knowledge base"}
)

print("Test collection created successfully.")

Test collection created successfully.


In [17]:
# Select the first 5 knowledge-base chunks

test_chunks = chunks_df.head(5)

print("Number of test chunks:", test_chunks.height)
print("Columns:", test_chunks.columns)

Number of test chunks: 5
Columns: ['chunk_id', 'organization', 'document', 'page_number', 'chunk_number', 'text', 'source_id', 'title', 'url']


In [18]:
# Prepare metadata for the test chunks

test_metadatas = [
    {
        "organization": str(row["organization"]),
        "document": str(row["document"]),
        "page_number": int(row["page_number"]),
        "chunk_number": int(row["chunk_number"]),
        "source_id": int(row["source_id"]),
        "title": str(row["title"]),
        "url": str(row["url"])
    }
    for row in test_chunks.to_dicts()
]

print("Metadata prepared successfully.")
print(test_metadatas[0])

Metadata prepared successfully.
{'organization': 'WHO', 'document': 'WHO BLEEDING AFTER BIRTH', 'page_number': 2, 'chunk_number': 1, 'source_id': 10, 'title': 'Bleeding After Birth: Course on Prevention Diagnosis and Treatment of Postpartum Haemorrhage', 'url': 'https://www.who.int/publications/i/item/9789240115835'}


In [19]:
# Add the first 5 chunks and their embeddings to ChromaDB

test_collection.add(
    ids=[str(x) for x in test_chunks["chunk_id"].to_list()],
    embeddings=loaded_embeddings[:5].tolist(),
    documents=test_chunks["text"].to_list(),
    metadatas=test_metadatas
)

print("5 test chunks added successfully.")

5 test chunks added successfully.


In [20]:
# Check how many chunks are currently stored

print("Chunks in test collection:", test_collection.count())

Chunks in test collection: 5


In [21]:
# Test semantic search

test_question = "What can cause heavy bleeding after childbirth?"

question_embedding = embedding_model.encode(
    test_question,
    convert_to_numpy=True,
    normalize_embeddings=True
)

results = test_collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

print("Search completed successfully.")
print(results.keys())

Search completed successfully.
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [22]:
# Display the retrieved chunks

for i, text in enumerate(results["documents"][0], start=1):
    print(f"\n--- Result {i} ---")
    print(text[:1000])


--- Result 1 ---
Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,

--- Result 2 ---
use of the WHO logo is not permitted. If you

In [23]:
# Remove the temporary test collection

chroma_client.delete_collection("pregnancy_knowledge_test")

print("Test collection deleted.")

Test collection deleted.


In [24]:
print(
    "Test collection exists:",
    any(
        collection.name == "pregnancy_knowledge_test"
        for collection in chroma_client.list_collections()
    )
)

Test collection exists: False


In [25]:
# Create the main pregnancy knowledge collection

knowledge_collection = chroma_client.get_or_create_collection(
    name="pregnancy_knowledge",
    metadata={
        "description": "WHO and UNICEF pregnancy and maternal health knowledge base"
    }
)

print("Main knowledge collection created successfully.")

Main knowledge collection created successfully.


In [26]:
# Add all knowledge-base chunks to ChromaDB in batches

batch_size = 100
total_chunks = chunks_df.height

for start in range(0, total_chunks, batch_size):
    end = min(start + batch_size, total_chunks)

    batch = chunks_df.slice(start, end - start)

    batch_embeddings = loaded_embeddings[start:end]

    batch_metadatas = [
        {
            "organization": str(row["organization"]),
            "document": str(row["document"]),
            "page_number": int(row["page_number"]),
            "chunk_number": int(row["chunk_number"]),
            "source_id": int(row["source_id"]),
            "title": str(row["title"]),
            "url": str(row["url"])
        }
        for row in batch.to_dicts()
    ]

    knowledge_collection.add(
        ids=[str(x) for x in batch["chunk_id"].to_list()],
        embeddings=batch_embeddings.tolist(),
        documents=batch["text"].to_list(),
        metadatas=batch_metadatas
    )

    print(f"Added chunks {start + 1}–{end} of {total_chunks}")

print("\nAll chunks added successfully.")

Added chunks 1–100 of 3596
Added chunks 101–200 of 3596
Added chunks 201–300 of 3596
Added chunks 301–400 of 3596
Added chunks 401–500 of 3596
Added chunks 501–600 of 3596
Added chunks 601–700 of 3596
Added chunks 701–800 of 3596
Added chunks 801–900 of 3596
Added chunks 901–1000 of 3596
Added chunks 1001–1100 of 3596
Added chunks 1101–1200 of 3596
Added chunks 1201–1300 of 3596
Added chunks 1301–1400 of 3596
Added chunks 1401–1500 of 3596
Added chunks 1501–1600 of 3596
Added chunks 1601–1700 of 3596
Added chunks 1701–1800 of 3596
Added chunks 1801–1900 of 3596
Added chunks 1901–2000 of 3596
Added chunks 2001–2100 of 3596
Added chunks 2101–2200 of 3596
Added chunks 2201–2300 of 3596
Added chunks 2301–2400 of 3596
Added chunks 2401–2500 of 3596
Added chunks 2501–2600 of 3596
Added chunks 2601–2700 of 3596
Added chunks 2701–2800 of 3596
Added chunks 2801–2900 of 3596
Added chunks 2901–3000 of 3596
Added chunks 3001–3100 of 3596
Added chunks 3101–3200 of 3596
Added chunks 3201–3300 of 359

In [27]:
print("Chunks in ChromaDB:", knowledge_collection.count())

Chunks in ChromaDB: 3596


In [28]:
assert knowledge_collection.count() == chunks_df.height

print("✓ All 3,596 chunks are stored correctly.")

✓ All 3,596 chunks are stored correctly.


In [29]:
# Function to test semantic retrieval

def search_knowledge_base(question, n_results=5):
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = knowledge_collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=n_results
    )

    return results

In [30]:
question = "What are the danger signs during pregnancy?"

results = search_knowledge_base(question, n_results=5)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"RESULT {i}")
    print(f"{'=' * 70}")
    print(f"Organization: {metadata['organization']}")
    print(f"Document: {metadata['document']}")
    print(f"Page: {metadata['page_number']}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {metadata['url']}")
    print("\nText:")
    print(document[:1500])


RESULT 1
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 165
Distance: 0.7339
Source: https://www.who.int/publications/b/31363

Text:
after birth.
When to seek care for danger signs
Go to hospital or health centre immediately, day or night, DO NOT wait, if any of the following signs:
 Vaginal bleeding has increased.
 Fits.
 Fast or difficult breathing.
 Fever and too weak to get out of bed.
 Severe headaches with blurred vision.
 Calf pain, redness or swelling; shortness of breath or chest pain.
Go to health centre as soon as possible if any of the following signs:
 Swollen, red or tender breasts or nipples.
 Problems urinating, or leaking.
 Increased pain or infection in the perineum.
 Infection in the area of the wound.
 Smelly vaginal discharge.
INFORMATION AND COUNSELLING SHEETS
M4Care for the mother after birth

RESULT 2
Organization: WHO
Document: Who recommendations on Antenatal care for positive pregnancy experience
Page: 92
Distan

In [31]:
question = "What are the signs that labour is starting?"

results = search_knowledge_base(question, n_results=5)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"RESULT {i}")
    print(f"{'=' * 70}")
    print(f"Organization: {metadata['organization']}")
    print(f"Document: {metadata['document']}")
    print(f"Page: {metadata['page_number']}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {metadata['url']}")
    print("\nText:")
    print(document[:1500])


RESULT 1
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 72
Distance: 1.0611
Source: https://www.who.int/publications/b/31363

Text:
FIRST STAGE OF LABOUR: IN ACTIVE LABOUR
Use this chart when the woman is IN ACTIVE LABOUR, when cervix dilated 4 cm or more.
MONITOR EVERY 30 MINUTES: MONITOR EVERY 4 HOURS:
 For emergency signs, using rapid assessment (RAM) B3-B7 .
 Frequency, intensity and duration of contractions.
 Fetal heart rate D14 .
 Mood and behaviour (distressed, anxious) D6 .
 Cervical dilatation D3 D15 .
 Unless indicated, do not do vaginal examination more frequently than every 4 hours.
 Temperature.
 Pulse B3 .
 Blood pressure D23 .
 Record findings regularly in Labour record and Partograph N4-N6 .
 Record time of rupture of membranes and colour of amniotic fluid.
 Give Supportive care D6-D7 .
 Never leave the woman alone.
ASSESS PROGRESS OF LABOUR TREAT AND ADVISE, IF REQUIRED
 Partograph passes to the right of ALERT LINE